# IRWA Final Project — Part 2: Indexing and Evaluation

**Group:** Xavier Gutiérrez (u233714), Pau Martínez (u232318), Mikel Areta (u232868)

We reuse the pre-processing from Part 1, build an inverted index, rank results with TF-IDF and evaluate the search engine.

In [1]:
import json, math, re, string
from collections import Counter, defaultdict

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

for res in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(res, quiet=True)

### Pre-processing (same as Part 1)
Documents and queries must be cleaned in exactly the same way, otherwise the query words would not match the indexed words.
Each product becomes one list of tokens: title + description + useful product_details + brand + sub_category.

In [2]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
PUNCT_TABLE = str.maketrans({c: " " for c in string.punctuation})


def preprocess(text):
    if not isinstance(text, str):
        return []
    text = text.encode("ascii", "ignore").decode("ascii").lower()
    text = re.sub(r"'s\b", "", text)                        # women's -> women
    text = re.sub(r"\bt[\s-]?shirts?\b", "tshirt", text)    # t-shirt / t shirt -> tshirt
    tokens = word_tokenize(text.translate(PUNCT_TABLE))
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1 and not any(c.isdigit() for c in t)]
    return [STEMMER.stem(t) for t in tokens]


with open("../../data/fashion_products_dataset.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

DROP_DETAIL_KEYS = {"Style Code", "Fabric Care", "Sales Package", "Country of Origin", "Pack of", "Reversible",
                    "Size", "Other Details", "Model Name", "Suitable For", "Brand Color", "Brand Fit", "Sleeve Type"}
CATEGORIES = [c for c in df["category"].unique() if " " in c]


def details_to_text(details):
    return " ".join(str(v) for d in details for k, v in d.items() if k not in DROP_DETAIL_KEYS)


def normalize_sub_category(value):   # 'Roy Clothing and Accessories' -> 'Clothing and Accessories'
    return next((c for c in CATEGORIES if value.endswith(c) and value != c), value)


df["sub_category"] = df["sub_category"].map(normalize_sub_category)
df["tokens"] = (df["title"].map(preprocess) + df["description"].map(preprocess)
                + df["product_details"].map(lambda d: preprocess(details_to_text(d)))
                + df["brand"].map(preprocess) + df["sub_category"].map(preprocess))
titles = dict(zip(df["pid"], df["title"]))
print("Documents:", len(df))

Documents: 28080


---
# Part 1: Indexing

## 1. Build inverted index
> After having pre-processed the data, you can then create the inverted index.
>
> **Important:** For this assignment, we will be using conjunctive queries (AND). This means that every returned document must contain all the words from the query in order to be considered a match.

The inverted index tells us, for every word, which documents contain it: `{term: [pid_1, pid_2, ...]}`.
We also save how many times each word appears in each document (`tf`), because TF-IDF needs it later.

In [3]:
index = defaultdict(list)   # term -> list of pids
tf = {}                     # pid  -> {term: count}

for pid, tokens in zip(df["pid"], df["tokens"]):
    tf[pid] = Counter(tokens)
    for term in tf[pid]:
        index[term].append(pid)

N = len(tf)
print("Vocabulary size:", len(index))
print("'sweatshirt' appears in", len(index["sweatshirt"]), "documents, e.g.", index["sweatshirt"][:3])

Vocabulary size: 5058
'sweatshirt' appears in 1417 documents, e.g. ['SWSFYD33P2XCBWFG', 'SWSFYD3KMGTBZEJZ', 'SWSFYD38Y2MAT9FQ']


**AND search:** a document is returned only if it contains *all* the query words, so we take the intersection of their document lists.

In [4]:
def conjunctive_search(query):
    terms = preprocess(query)
    if not terms:
        return set()
    return set.intersection(*[set(index.get(t, [])) for t in terms])


for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue"]:
    print(f"{q!r}: {len(conjunctive_search(q))} documents")

'women full sleeve sweatshirt cotton': 467 documents
'men slim jeans blue': 216 documents


## 2. Propose test queries
> Define five queries that will be used to evaluate your search engine. (Be creative 😉)
>
> HINT: How to choose the queries? The selection of the queries is up to you, but it's suggested to select terms based on the popularity (keywords ranked by term frequencies or by TF-IDF, etc).

First we look at the most popular words (in how many documents each word appears):

In [5]:
doc_freq = pd.Series({term: len(pids) for term, pids in index.items()}).sort_values(ascending=False)
doc_freq.head(30).to_dict()

{'cotton': 19610,
 'sleev': 17909,
 'topwear': 15310,
 'neck': 14578,
 'men': 14141,
 'women': 13699,
 'blend': 13593,
 'regular': 13089,
 'tshirt': 12902,
 'solid': 11989,
 'round': 9497,
 'print': 9484,
 'casual': 9480,
 'half': 8062,
 'fit': 7777,
 'wear': 7717,
 'full': 7156,
 'blue': 6978,
 'comfort': 6845,
 'pack': 5924,
 'black': 5898,
 'pure': 5498,
 'slim': 5422,
 'fabric': 5059,
 'look': 5050,
 'design': 4948,
 'short': 4941,
 'shirt': 4809,
 'multicolor': 4752,
 'made': 4471}

We mixed popular words (gender, fit, colour, *shirt*, *sports*) with more specific ones (*hooded*, *running*, *kurta*), so the queries look like real searches and return a number of results we can judge.

| Query | Why |
|---|---|
| women hooded jacket | a clothing type + an attribute (*hooded*) |
| men running shoes | footwear, a much smaller category than topwear |
| women yellow kurta | colour + an Indian garment, very specific |
| men regular fit formal shirt | long query made of very common words → tests IDF |
| men sports shorts | *sports* appears in many other products → tests ranking |

In [6]:
test_queries = {
    3: "women hooded jacket",
    4: "men running shoes",
    5: "women yellow kurta",
    6: "men regular fit formal shirt",
    7: "men sports shorts",
}
for qid, q in test_queries.items():
    print(f"{qid}. {q:30} -> {len(conjunctive_search(q))} results")

3. women hooded jacket            -> 80 results
4. men running shoes              -> 76 results
5. women yellow kurta             -> 39 results
6. men regular fit formal shirt   -> 218 results
7. men sports shorts              -> 196 results


## 3. Rank your results
> Implement the TF-IDF algorithm and provide ranking-based results.

Each word gets a weight that is high when it appears often in the document (**tf**) and is rare in the collection (**idf**):

$$w_{t,d} = (1 + \log tf_{t,d}) \cdot \log\frac{N}{df_t}$$

The query is turned into a vector in the same way, and the score is the **cosine similarity** between both vectors (so long documents are not favoured just for being long):

$$score(q,d) = \frac{\sum_t w_{t,q}\, w_{t,d}}{\lVert q \rVert\, \lVert d \rVert}$$

In [7]:
idf = {term: math.log(N / len(pids)) for term, pids in index.items()}


def weight(count, term):
    return (1 + math.log(count)) * idf[term]


# length of every document vector (computed once)
doc_norm = {pid: math.sqrt(sum(weight(c, t) ** 2 for t, c in counts.items())) or 1.0 for pid, counts in tf.items()}


def tfidf_score(query_terms, pid):
    q = Counter(t for t in query_terms if t in idf)
    q_norm = math.sqrt(sum(weight(c, t) ** 2 for t, c in q.items())) or 1.0
    dot = sum(weight(c, t) * weight(tf[pid][t], t) for t, c in q.items() if t in tf[pid])
    return dot / (q_norm * doc_norm[pid])


def search(query, top_k=None):
    """AND search + TF-IDF ranking (ties broken by pid)."""
    terms = preprocess(query)
    results = [(pid, tfidf_score(terms, pid)) for pid in conjunctive_search(query)]
    results.sort(key=lambda x: (-x[1], x[0]))
    results = results[:top_k]
    return pd.DataFrame({"rank": range(1, len(results) + 1),
                         "pid": [p for p, _ in results],
                         "score": [round(s, 4) for _, s in results],
                         "title": [titles[p] for p, _ in results]})


for q in test_queries.values():
    print("\nQuery:", q)
    print(search(q, top_k=5).to_string(index=False))


Query: women hooded jacket
 rank              pid  score                                 title
    1 JCKFVEVFZ6BGWHFZ 0.6129 Full Sleeve Solid Women Sports Jacket
    2 JCKFUFGZZSPGSYSS 0.5916 Full Sleeve Solid Women Casual Jacket
    3 JCKFRAJTUACHZVHP 0.5751 Full Sleeve Solid Women Casual Jacket
    4 JCKFPRBPRGG6BCSA 0.5710 Full Sleeve Solid Women Casual Jacket
    5 JCKFRAJTVVHJJGGW 0.5690 Full Sleeve Solid Women Casual Jacket

Query: men running shoes
 rank              pid  score                                       title
    1 SHOEP5GUQYYFKYSN 0.6615            Running Shoes For Women  (White)
    2 SHOESS9ZEZ6TEQUJ 0.6493         Running Shoes For Men  (Multicolor)
    3 SHOESS9ZGMH7CURM 0.6493         Running Shoes For Men  (Multicolor)
    4 SHOEKNRJGEYJCDQK 0.5076  SPRINT Running Shoes For Men  (Blue, Grey)
    5 SHOEKNRJ34UCMAPA 0.5059 SPRINT Running Shoes For Men  (Black, Grey)

Query: women yellow kurta
 rank              pid  score                                      

---
# Part 2: Evaluation

## 1. Implement the following evaluation metrics
> Implement the following evaluation metrics to assess the effectiveness of your retrieval solutions. These metrics will help you measure how well your system retrieves relevant documents for each query:
> i. Precision@K (P@K) · ii. Recall@K (R@K) · iii. Average Precision@K (P@K) · iv. F1-Score@K · v. Mean Average Precision (MAP) · vi. Mean Reciprocal Rank (MRR) · vii. Normalized Discounted Cumulative Gain (NDCG)

Every function receives `rels`: the 1/0 labels of the results **in ranking order**, and `n_rel`: how many relevant documents exist for the query.

| Metric | Formula |
|---|---|
| P@K | relevant in top K / K |
| R@K | relevant in top K / n_rel |
| AP@K | average of P@i at each position i ≤ K that holds a relevant doc (divided by min(n_rel, K)) |
| F1@K | 2·P·R / (P + R) |
| MAP | mean of AP over all queries |
| MRR | mean of 1 / (position of the first relevant result) |
| NDCG@K | DCG / ideal DCG, with $DCG@K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i+1)}$ |

In [8]:
def precision_at_k(rels, k):
    return sum(rels[:k]) / k


def recall_at_k(rels, k, n_rel):
    return sum(rels[:k]) / n_rel if n_rel else 0.0


def avg_precision_at_k(rels, k, n_rel):
    hits, total = 0, 0.0
    for i, r in enumerate(rels[:k], start=1):
        if r:
            hits += 1
            total += hits / i
    return total / min(n_rel, k) if n_rel else 0.0


def f1_at_k(rels, k, n_rel):
    p, r = precision_at_k(rels, k), recall_at_k(rels, k, n_rel)
    return 2 * p * r / (p + r) if p + r else 0.0


def reciprocal_rank(rels, k):
    for i, r in enumerate(rels[:k], start=1):
        if r:
            return 1 / i
    return 0.0


def ndcg_at_k(rels, k, n_rel):
    dcg = sum(r / math.log2(i + 1) for i, r in enumerate(rels[:k], start=1))
    ideal = sum(1 / math.log2(i + 1) for i in range(1, min(n_rel, k) + 1))
    return dcg / ideal if ideal else 0.0


# MAP and MRR are just the mean of AP and RR over all queries
def mean_average_precision(all_rels, k, all_n_rel):
    return np.mean([avg_precision_at_k(r, k, n) for r, n in zip(all_rels, all_n_rel)])


def mean_reciprocal_rank(all_rels, k):
    return np.mean([reciprocal_rank(r, k) for r in all_rels])


# quick check with a toy ranking (3 relevant docs in total)
toy = [1, 0, 1, 0, 0, 1]
print(precision_at_k(toy, 3), recall_at_k(toy, 3, 3), avg_precision_at_k(toy, 6, 3),
      f1_at_k(toy, 3, 3), reciprocal_rank(toy, 6), round(ndcg_at_k(toy, 6, 3), 3))

0.6666666666666666 0.6666666666666666 0.7222222222222222 0.6666666666666666 1.0 0.871


In [9]:
K = 10


def metrics_table(all_rels, all_n_rel, names, k=K):
    """All metrics per query, rounded to 3 decimals, plus MAP and MRR."""
    table = pd.DataFrame([{"query": name,
                           f"P@{k}": precision_at_k(r, k), f"R@{k}": recall_at_k(r, k, n),
                           f"AP@{k}": avg_precision_at_k(r, k, n), f"F1@{k}": f1_at_k(r, k, n),
                           f"RR@{k}": reciprocal_rank(r, k), f"NDCG@{k}": ndcg_at_k(r, k, n)}
                          for name, r, n in zip(names, all_rels, all_n_rel)]).set_index("query").round(3)
    summary = pd.Series({f"MAP@{k}": mean_average_precision(all_rels, k, all_n_rel),
                         f"MRR@{k}": mean_reciprocal_rank(all_rels, k)}).round(3)
    return table, summary

## 2. Apply the evaluation metrics to validation_labels.csv
> Apply the evaluation metrics you have implemented to the search results and relevance judgments provided in validation_labels.csv for the predefined queries. When reporting evaluation results, provide only numeric values, rounded to three decimal places. Do not include textual explanations or additional statistics in this section.
> a. Query 1: women full sleeve sweatshirt cotton
> b. Query 2: men slim jeans blue

In [10]:
# Keep, in ranking order, the results that have a label; relevant labelled docs not returned count as missed
labels = pd.read_csv("../../data/validation_labels.csv")
validation_queries = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}

val_rels, val_n_rel = [], []
for qid, q in validation_queries.items():
    judged = labels[labels["query_id"] == qid].set_index("pid")["labels"]
    ranking = search(q)
    ranking = ranking[ranking["pid"].isin(judged.index)]
    val_rels.append([int(judged[p]) for p in ranking["pid"]])
    val_n_rel.append(int(judged.sum()))

val_table, val_summary = metrics_table(val_rels, val_n_rel, list(validation_queries.values()))
val_table

,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
query,,,,,,
women full sleeve sweatshirt cotton,1.0,0.769,1.000,0.87,1.0,1.000
men slim jeans blue,0.6,0.600,0.569,0.60,1.0,0.716


In [11]:
val_summary

MAP@10    0.785
MRR@10    1.000
dtype: float64

## 3. You will act as expert judges by establishing the ground truth for each document and query

### a. Relevance labels for our test queries
> For the test queries you defined in Part 1, Step 2 during indexing, assign a binary relevance label to each document: 1 if the document is relevant to the query, or 0 if it is not.

We judged the **top 20 results** of each query by reading their title, description and product details. The labels are in `my_labels.csv`.

A product is relevant (1) only if it is the right **type** of product, for the right **gender**, with the requested **attribute** (hooded, yellow, formal…).
Since we cannot judge all 28,080 products, the number of relevant documents of each query is the number of 1s among its 20 judged results.

In [12]:
# The results we judged (top 20 of each query):
# for q in test_queries.values(): display(search(q, top_k=20))

my_labels = pd.read_csv("my_labels.csv").sort_values(["query_id", "rank"])
my_labels.groupby("query")["label"].agg(relevant="sum", judged="count")

,relevant,judged
query,,
men regular fit formal shirt,20,20
men running shoes,11,20
men sports shorts,20,20
women hooded jacket,14,20
women yellow kurta,13,20


In [13]:
my_rels = [g["label"].tolist() for _, g in my_labels.groupby("query_id")]
my_n_rel = [sum(r) for r in my_rels]

my_table, my_summary = metrics_table(my_rels, my_n_rel, list(test_queries.values()))
my_table

,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
query,,,,,,
women hooded jacket,0.7,0.500,0.700,0.583,1.0,0.801
men running shoes,0.8,0.727,0.598,0.762,0.5,0.710
women yellow kurta,0.6,0.462,0.600,0.522,1.0,0.727
men regular fit formal shirt,1.0,0.500,1.000,0.667,1.0,1.000
men sports shorts,1.0,0.500,1.000,0.667,1.0,1.000


In [14]:
my_summary

MAP@10    0.78
MRR@10    0.90
dtype: float64

### b. Comment on each of the evaluation metrics
> Comment on each of the evaluation metrics, stating how they differ, and which information gives each of them. Analyze your results.

| Metric | What it tells us | What it doesn't see |
|---|---|---|
| **P@K** | How many of the first K results are good | The order inside the top K and the relevant docs we missed |
| **R@K** | How many of all the relevant docs we found in the top K | The bad results; it can only go up with K, while P@K usually goes down |
| **AP@K** | Like precision, but rewards putting the relevant docs **at the top** | — (it does care about order) |
| **F1@K** | A balance of P@K and R@K: only high if both are high | The order |
| **MAP** | One number for the ranking quality of the whole system | Differences between queries (an easy query can hide a hard one) |
| **MRR** | How soon the **first** good result appears (useful when you only want one product) | Everything after that first good result |
| **NDCG@K** | How close the ranking is to the perfect order (1 = perfect); results lower down count less | — (it also supports graded labels like 0/1/2) |

**What our results show**

- **Query 1** works perfectly: the first 10 results are all relevant (P@10 = AP@10 = NDCG@10 = 1). Recall is 0.769 only because there are 13 relevant documents and just 10 fit in the top 10.
- **Query 2** is much worse (P@10 = 0.6, AP@10 = 0.569). The AND search only returns 8 of the 20 labelled products: the file marks some **women's** jeans as relevant, but they don't contain *men*, so we can never find them. MRR is still 1 because the first result is correct, which shows that MRR can hide problems further down.
- **men regular fit formal shirt** and **men sports shorts** are easy (all metrics = 1 except recall): the query words are exactly the words used in the titles.
- **men running shoes** has a good P@10 (0.8) but the worst RR (0.5) and AP (0.598): the **first** result is a women's shoe. P@10 doesn't notice this, AP, RR and NDCG do. It happens because that product's sub_category is *"Men's Footwear"*.
- **women hooded jacket** (P@10 = 0.7) brings hooded **sweatshirts** at positions 8–11, because their description literally says *"Full Sleeve Hooded Jacket"*.
- **women yellow kurta** (P@10 = 0.6) mixes single kurtas with **kurta + pyjama sets**, which contain the same words.
- Recall is ≤ 0.5 for most of our queries just because each one has 11–20 relevant documents and only 10 fit in the top 10. That's why P and R should be read together (F1).
- **Overall:** MAP is similar in both sets (0.785 and 0.780) and MRR is high (1.0 and 0.9). The system usually puts a good product first, but the quality of the rest of the ranking changes a lot from one query to another.

### c. Main problems of the current system and how to solve them
> Analyze the current search system and identify its main problems or limitations. For each issue you find, propose possible ways to resolve it. Consider aspects such as retrieval accuracy, ranking quality, handling of different field types, query formulation, and indexing strategies.

| Aspect | Problem | Example | How to fix it |
|---|---|---|---|
| Query formulation | AND is too strict: if one word is missing, the product is lost | Query 2 only returns 8 of its 20 labelled products | Rank with OR, or use AND and fall back to OR when there are few results |
| Field types | All fields count the same | A word in a long description weighs as much as one in the title | Give more weight to the title than to the description (e.g. BM25F) |
| Retrieval accuracy | Gender is treated as any other word | A women's shoe is 1st for *men running shoes* (sub_category *Men's Footwear*) | Create a gender field from *Ideal For* and the title, and use it as a filter |
| Retrieval accuracy | The system doesn't know the product type | Sweatshirts for *hooded jacket*; kurta + pyjama sets for *kurta* | Boost the product type (*Generic Name*, last word of the title) |
| Ranking quality | Simple TF-IDF and many ties (78.8% of titles are repeated) | Two shoes with exactly 0.6493 in query 4 | Use BM25; break ties with rating, discount and stock |
| Query formulation | No synonyms or typo handling | *hoodie* vs *hooded*, *trainers* vs *shoes* | Synonym expansion, spelling correction, word embeddings |
| Indexing strategy | Words are indexed one by one | *full sleeve* is treated as two separate words | Index bigrams or word positions (phrase search) |
| Field types | Numeric fields are not used | Out-of-stock products are ranked like the rest | Use price, rating and stock as filters and ranking signals |
| Evaluation | Small ground truth judged by us | Only 20 judged documents per query | More queries, more judges, graded labels (0/1/2) for NDCG |